In [1]:
from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.systems.rcf_oil_purification import create_rcf_oil_purification_system
from lignin_saf.systems.monomer_purification import create_monomer_purification_system
from lignin_saf.systems.ligsaf_utilities import create_rcf_utilities_system
from lignin_saf.systems.cellulosic_ethanol import create_cellulosic_ethanol_system
from lignin_saf.cellulosic_tea import create_cellulosic_ethanol_tea
from lignin_saf.ligsaf_settings import solvolysis_parameters

from biosteam import main_flowsheet as F
import biosteam as bst
import pandas as pd
import numpy as np
import thermosteam as tmo



In [2]:
# Code just to increase the number of display units for the various components
tmo.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
tmo.MultiStream.display_units.N = 40  
bst.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
bst.MultiStream.display_units.N = 40  

In [3]:
chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7   # 2016 USD basis

# Poplar group must be defined before creating any stream that references it
chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('RCF_POPLAR_IN',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])

# ── Area 200: RCF process ──────────────────────────────────────────────────
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()



rcf_system.simulate()
rcf_system.show()

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: RCF_PUMP1> no pump type available at current power (2.34e+03 hp), head (3.2e+03 ft), kinematic viscosity (5.83e-07 m2/s), and NPSH (4.18 ft); assuming centrigugal pump
  warn(f'{repr(self)} no pump type available at current power '
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:409: CostWarning: <SolvolysisReactor: RCF_RXR1> Vertical vessel length (58.75 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1241: CostWarning: <IsentropicCompressor: RCF_COMP1> power (1.48

System: RCF_System
Highest convergence error among components in recycle
streams {RCF_HX4-0, RCF_PUMP2-0} after 1 loops:
- flow rate   8.96e-11 kmol/hr (0.005%)
- temperature 7.00e-04 K (0.00022%)
ins...
[0] RCF_CAT_IN  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): NiC  2.28
[1] RCF_H2_IN  
    phase: 'g', T: 353.15 K, P: 3e+06 Pa
    flow (kmol/hr): Hydrogen  771
[2] RCF_POPLAR_IN  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water     925
                    Sucrose   0.243
                    Extract   7.4
                    Acetate   48.6
                    Ash       1e+03
                    Lignin    156
                    Glucan    238
                    Xylan     84.5
                    Arabinan  1.26
                    Mannan    19
                    Galactan  7.2
[3] RCF_MEOH_IN  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Methanol  1.51e+03
[4] RCF_H2O_IN  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/h

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1241: CostWarning: <IsentropicCompressor: RCF_COMP1> power (0 hp) is out of bounds (10 to 750 hp) for cost correlation
  self._cost(**cost_kwargs) if cost_kwargs else self._cost()


In [4]:
methanol_decomposition_rxn = bst.ParallelReaction([
    bst.Reaction('Methanol,l -> Methane,g', reactant='Methanol', phases='lg',
                    X=solvolysis_parameters['MeOH_CH4'], basis='wt', correct_atomic_balance=False),
    bst.Reaction('Methanol,l -> CO,g', reactant='Methanol', phases='lg',
                    X=solvolysis_parameters['MeOH_CO'], basis='wt', correct_atomic_balance=False),
])


In [5]:
methanol = bst.Stream('Methanol', phase = 'l', Methanol = 100, units = 'kg/hr')

In [6]:
methanol_decomposition_rxn(methanol)

ValueError: reaction and stream phases do not match

In [ ]:
chems['Acetate']

Chemical: Acetate (phase_ref='l')
[Names]  CAS: Acetate
         InChI: None
         InChI_key: None
         common_name: None
         iupac_name: ()
         pubchemid: None
         smiles: None
         formula: C2H4O2
[Groups] Dortmund: <1CH3, 1COOH>
         UNIFAC: <1CH3, 1COOH>
         PSRK: <1CH3, 1COOH>
         NIST: <Empty>
[Data]   MW: 60.052 g/mol
         Tm: 289.85 K
         Tb: 391.05 K
         Tt: 289.69 K
         Tc: 590.7 K
         Pt: 1267.7 Pa
         Pc: 5.78e+06 Pa
         Vc: 0.000171 m^3/mol
         Hf: -4.8358e+05 J/mol
         S0: 159.8 J/K/mol
         LHV: 7.87e+05 J/mol
         HHV: 8.7502e+05 J/mol
         Hfus: 11730 J/mol
         Sfus: None
         omega: 0.4218
         dipole: 1.7 Debye
         similarity_variable: 0.13322
         iscyclic_aliphatic: 0
         combustion: {'CO2': 2, 'O2'...


In [ ]:
chems['AceticAcid']

Chemical: AceticAcid (phase_ref='l')
[Names]  CAS: 64-19-7
         InChI: C2H4O2/c1-2(3)4/h1H3...
         InChI_key: QTBSBXVTEAMEQO-U...
         common_name: acetic acid
         iupac_name: ('ethanoic acid...
         pubchemid: 176
         smiles: CC(=O)O
         formula: C2H4O2
[Groups] Dortmund: <1CH3, 1COOH>
         UNIFAC: <1CH3, 1COOH>
         PSRK: <1CH3, 1COOH>
         NIST: <Empty>
[Data]   MW: 60.052 g/mol
         Tm: 289.85 K
         Tb: 391.05 K
         Tt: 289.69 K
         Tc: 590.7 K
         Pt: 1267.7 Pa
         Pc: 5.78e+06 Pa
         Vc: 0.000171 m^3/mol
         Hf: -4.8358e+05 J/mol
         S0: 159.8 J/K/mol
         LHV: 7.87e+05 J/mol
         HHV: 8.7502e+05 J/mol
         Hfus: 11730 J/mol
         Sfus: None
         omega: 0.4218
         dipole: 1.7 Debye
         similarity_variable: 0.13322
         iscyclic_aliphatic: 0
         combustion: {'CO2': 2, 'O2'...
